In [ ]:
%load_ext autoreload
%autoreload 2

from math import pi
from glob import glob

import numpy as np
import matplotlib.pyplot as plt

import ast

import pandas as pd

In [ ]:
from matplotlib.colors import LogNorm
plt.rcParams.update({'font.size': 18})

In [ ]:
from numpy.random import normal

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
import numpy.random as random

### Setup functions for Minimizer

In [ ]:
# move randomly to a new point within a max distance of "d"
def RandomJump(x0,d):
    d0 = np.random.random() * 2*d - d
    return x0+d0

In [ ]:
# "time-decay" probability of jumping away from local area
def Shift(timetracker,timeconstant):
    # exponential f(x) = e^(-timetracker/timeconstant)
    # max value is 1
    # generate a random number from 0 to 1
    p = random.random()
    # make a big jump? depends on expoential
    v = np.exp(-timetracker/timeconstant)
    if (p < v): return True
    return False

In [ ]:
# have we converged? check if result is stable
def ConvergeCheck(m1,m2,MDIFF):
    if (np.abs(m1-m2) < MDIFF):
        return True
    return False

In [ ]:
# shrink scan radius over time
def ShrinkRadius(timetracker,timeconstant,R0):
    return R0 * np.exp(-timetracker/timeconstant)

### Find Minimum of parabolic function

In [ ]:
def PARABOLA(x,b=3.0):
    return b * x**2

In [ ]:
# step 1 -> initial guess
x0 = random.random()*10

# step 2 -> iteratively find a minimum
CONVERGE = False # have we converged?
METRIC = 1e-2 # convergence requirement
TIME = 0 # number of iterations is proxy for time
TAU_RADIUS = 50. # time-scale for radius to shrink
TAU_SHIFT = 10. # time-scale for jump to new location
R0 = 3.0 # initial radius to scan
TMAX = 10 # maximum number of steps to try before giving up
VERBOSE = True # verbosity flag
while (CONVERGE == False):
    
    TIME += 1 # step up in time
    RADIUS = R0
    
    if(VERBOSE): print('\n\ntime step %i \t x0: %.02f'%(TIME,x0))
    
    # have we run out of time?
    if (TIME > TMAX):
        if (VERBOSE): print('Ran out of time!')
        break
    
    # jump to new location
    x1 = RandomJump(x0,RADIUS)
    if(VERBOSE): print('Random jump to %.02f'%x1)
    
    # chheck if we have converged
    F0 = PARABOLA(x0)
    F1 = PARABOLA(x1)
    if(VERBOSE): print('x0: %.02f -> x1: %.02f with values %.02f & %.02f'%(x0,x1,F0,F1))
    if (ConvergeCheck(F0,F1,METRIC) == True):
        # we have converged! Exit
        if(VERBOSE): print('diff < METRIC %.03f -> CONVERGE!')
        CONVERGE = True
        break
        
    # if not, should we move to the new point?
    if (F1 < F0):
        if(VERBOSE): print('Moving in Right Direction! Jump to x1')
        x0 = x1
    
    # update RADIUS based on time passed
    RADIUS = ShrinkRadius(TIME,TAU_RADIUS,R0)
    
    # decide if to jump to a completely new location?
    if (Shift(TIME,TAU_SHIFT) == True):
        x0 = np.random.random()*10
        if(VERBOSE): print('Shift to a new location! New x0 is %.02f'%x0)

In [ ]:
import time
from IPython.display import display, clear_output

In [ ]:
# step 1 -> initial guess
x0 = random.random()*20-10

# step 2 -> iteratively find a minimum
CONVERGE = False # have we converged?
METRIC = 1e-2 # convergence requirement
TIME = 0 # number of iterations is proxy for time
TAU_RADIUS = 5. # time-scale for radius to shrink
TAU_SHIFT = 10. # time-scale for jump to new location
R0 = 3.0 # initial radius to scan
TMAX = 100 # maximum number of steps to try before giving up
VERBOSE = False # verbosity flag

fig, ax = plt.subplots()
xvals = np.linspace(-10,10,1000)
PARABOLA_V = np.vectorize(PARABOLA)
yvals = PARABOLA_V(xvals)

RADIUS = R0

x_history = []
f_history = []

while (CONVERGE == False):
    
    TIME += 1 # step up in time
    
    # plotting
    ax.cla()
    ax.plot(xvals,yvals,'k--',label='P(shift) = %.02f'%(np.exp(-TIME/TAU_SHIFT)))
    ax.axvspan(x0-RADIUS, x0+RADIUS, color='gray', alpha=0.5,label='R0 = %.02f'%RADIUS)
    
    # decide if to jump to a completely new location?
    if (Shift(TIME,TAU_SHIFT) == True):
        x0old = x0
        x0 = np.random.random()*10
        ax.plot([x0old,x0],[PARABOLA(x0old),PARABOLA(x0)],'o-',color='g',label='iteration %i'%TIME)
        if(VERBOSE): 
            print('Shift to a new location! New x0 is %.02f'%x0)
            continue
    
    if(VERBOSE): print('\n\ntime step %i \t x0: %.02f'%(TIME,x0))
    
    # have we run out of time?
    if (TIME > TMAX):
        if (VERBOSE): print('Ran out of time!')
        break
    
    # jump to new location
    x1 = RandomJump(x0,RADIUS)
    if(VERBOSE): print('Random jump to %.02f'%x1)
    
    # chheck if we have converged
    F0 = PARABOLA(x0)
    F1 = PARABOLA(x1)
    if(VERBOSE): print('x0: %.02f -> x1: %.02f with values %.02f & %.02f'%(x0,x1,F0,F1))
    if(VERBOSE): print('diff: %.02f'%np.abs(F0-F1))
        
    # plotting            
    x_history.append(x0)
    f_history.append(F0)
    ax.plot(x_history,f_history,'-',color='orange',lw=1)
    ax.plot([x0,x1],[F0,F1],'o-',color='b')
        
    if (ConvergeCheck(F0,F1,METRIC) == True):
        # we have converged! Exit
        if(VERBOSE): print('diff < METRIC %.03f -> CONVERGE!')
        CONVERGE = True
        break
        
    # if not, should we move to the new point?
    if (F1 < F0):
        if(VERBOSE): print('Moving in Right Direction! Jump to x1')
        ax.plot([x1],[F1],'ro',markersize=5)
        x0 = x1
        
    # plotting   
    ax.legend(loc=1,fontsize=14)
    
    # update RADIUS based on time passed
    RADIUS = ShrinkRadius(TIME,TAU_RADIUS,R0)
    

            
    # Update the display
    display(fig)
    clear_output(wait=True)
    time.sleep(1)

In [ ]:
def WAIVY_PARABOLA(x,b=3.0):
    s = 30*np.sin(3*x)
    return b * x**2 + s

In [ ]:
# step 1 -> initial guess
x0 = random.random()*20-10

# step 2 -> iteratively find a minimum
CONVERGE = False # have we converged?
METRIC = 1e-2 # convergence requirement
TIME = 0 # number of iterations is proxy for time
TAU_RADIUS = 5. # time-scale for radius to shrink
TAU_SHIFT = 1. # time-scale for jump to new location
R0 = 3.0 # initial radius to scan
TMAX = 100 # maximum number of steps to try before giving up
VERBOSE = False # verbosity flag

fig, ax = plt.subplots()
xvals = np.linspace(-10,10,1000)
WAIVY_PARABOLA_V = np.vectorize(WAIVY_PARABOLA)
yvals = WAIVY_PARABOLA_V(xvals)

RADIUS = R0

x_history = []
f_history = []

while (CONVERGE == False):
    
    TIME += 1 # step up in time
    
    # plotting
    ax.cla()
    ax.plot(xvals,yvals,'k--',label='P(shift) = %.02f'%(np.exp(-TIME/TAU_SHIFT)))
    ax.axvspan(x0-RADIUS, x0+RADIUS, color='gray', alpha=0.5,label='R0 = %.02f'%RADIUS)
    
    # decide if to jump to a completely new location?
    if (Shift(TIME,TAU_SHIFT) == True):
        x0old = x0
        x0 = np.random.random()*10
        ax.plot([x0old,x0],[WAIVY_PARABOLA(x0old),WAIVY_PARABOLA(x0)],'o-',color='g',label='iteration %i'%TIME)
        if(VERBOSE): 
            print('Shift to a new location! New x0 is %.02f'%x0)
            continue
    
    if(VERBOSE): print('\n\ntime step %i \t x0: %.02f'%(TIME,x0))
    
    # have we run out of time?
    if (TIME > TMAX):
        if (VERBOSE): print('Ran out of time!')
        break
    
    # jump to new location
    x1 = RandomJump(x0,RADIUS)
    if(VERBOSE): print('Random jump to %.02f'%x1)
    
    # chheck if we have converged
    F0 = WAIVY_PARABOLA(x0)
    F1 = WAIVY_PARABOLA(x1)
    if(VERBOSE): print('x0: %.02f -> x1: %.02f with values %.02f & %.02f'%(x0,x1,F0,F1))
    if(VERBOSE): print('diff: %.02f'%np.abs(F0-F1))
        
    # plotting            
    x_history.append(x0)
    f_history.append(F0)
    ax.plot(x_history,f_history,'-',color='orange',lw=1)
    ax.plot([x0,x1],[F0,F1],'o-',color='b')
        
    if (ConvergeCheck(F0,F1,METRIC) == True):
        # we have converged! Exit
        if(VERBOSE): print('diff < METRIC %.03f -> CONVERGE!')
        CONVERGE = True
        break
        
    # if not, should we move to the new point?
    if (F1 < F0):
        if(VERBOSE): print('Moving in Right Direction! Jump to x1')
        ax.plot([x1],[F1],'ro',markersize=5)
        x0 = x1
        
    # plotting   
    ax.legend(loc=1,fontsize=14)
    
    # update RADIUS based on time passed
    RADIUS = ShrinkRadius(TIME,TAU_RADIUS,R0)
    

            
    # Update the display
    display(fig)
    clear_output(wait=True)
    time.sleep(1)